# Iris Dataset: Detailed Pandas Exploration

This notebook uses the real Iris dataset from the workspace to explore more pandas basics in a practical way.

In [8]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda value: f'{value:.3f}')

## 1. Load the dataset

The CSV in the workspace has no header row, so we assign meaningful column names while loading it.

In [9]:
candidate_paths = [
    Path('../15-04-26/data/iris.csv'),
    Path('python_programs/15-04-26/data/iris.csv'),
    Path('../../python_programs/15-04-26/data/iris.csv'),
]

iris_path = next((candidate for candidate in candidate_paths if candidate.exists()), None)
if iris_path is None:
    raise FileNotFoundError('Could not locate iris.csv in the workspace.')

iris = pd.read_csv(
    iris_path,
    header=None,
    names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
)

print(f'Loaded dataset from: {iris_path}')
print(f'Rows: {iris.shape[0]}')
print(f'Columns: {iris.shape[1]}')
display(iris.head())

Loaded dataset from: ..\15-04-26\data\iris.csv
Rows: 150
Columns: 5


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.100,3.500,1.400,0.200,Iris-setosa
1,4.900,3.000,1.400,0.200,Iris-setosa
2,4.700,3.200,1.300,0.200,Iris-setosa
3,4.600,3.100,1.500,0.200,Iris-setosa
4,5.000,3.600,1.400,0.200,Iris-setosa


## 2. Quick inspection

These are the first checks to run on any DataFrame.

In [10]:
display(iris.tail())
display(iris.dtypes.to_frame('dtype'))
display(iris.isna().sum().to_frame('missing_count'))
display(iris.duplicated().sum())

,sepal_length,sepal_width,petal_length,petal_width,species
145,6.700,3.000,5.200,2.300,Iris-virginica
146,6.300,2.500,5.000,1.900,Iris-virginica
147,6.500,3.000,5.200,2.000,Iris-virginica
148,6.200,3.400,5.400,2.300,Iris-virginica
149,5.900,3.000,5.100,1.800,Iris-virginica


,dtype
sepal_length,float64
sepal_width,float64
petal_length,float64
petal_width,float64
species,object


,missing_count
sepal_length,0
sepal_width,0
petal_length,0
petal_width,0
species,0


3

In [11]:
iris.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


## 3. Descriptive statistics

`describe()` gives a compact summary of numeric columns.

In [12]:
display(iris.describe())
display(iris.describe(include='object'))
display(iris['species'].value_counts().to_frame('count'))

,sepal_length,sepal_width,petal_length,petal_width
count,150.000,150.000,150.000,150.000
mean,5.843,3.054,3.759,1.199
std,0.828,0.434,1.764,0.763
min,4.300,2.000,1.000,0.100
25%,5.100,2.800,1.600,0.300
50%,5.800,3.000,4.350,1.300
75%,6.400,3.300,5.100,1.800
max,7.900,4.400,6.900,2.500


,species
count,150
unique,3
top,Iris-setosa
freq,50


,count
species,
Iris-setosa,50
Iris-versicolor,50
Iris-virginica,50


## 4. Select, filter, and compare rows

Here we use column selection, `loc`, `iloc`, and boolean filtering.

In [13]:
display(iris[['sepal_length', 'petal_length', 'species']].head(10))
display(iris.loc[iris['species'] == 'Iris-setosa'].head())
display(iris.iloc[:5, :3])
display(iris.query('petal_length > 5 and petal_width > 1.5').head())

,sepal_length,petal_length,species
0,5.100,1.400,Iris-setosa
1,4.900,1.400,Iris-setosa
2,4.700,1.300,Iris-setosa
3,4.600,1.500,Iris-setosa
4,5.000,1.400,Iris-setosa
5,5.400,1.700,Iris-setosa
6,4.600,1.400,Iris-setosa
7,5.000,1.500,Iris-setosa
8,4.400,1.400,Iris-setosa
9,4.900,1.500,Iris-setosa


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.100,3.500,1.400,0.200,Iris-setosa
1,4.900,3.000,1.400,0.200,Iris-setosa
2,4.700,3.200,1.300,0.200,Iris-setosa
3,4.600,3.100,1.500,0.200,Iris-setosa
4,5.000,3.600,1.400,0.200,Iris-setosa


,sepal_length,sepal_width,petal_length
0,5.100,3.500,1.400
1,4.900,3.000,1.400
2,4.700,3.200,1.300
3,4.600,3.100,1.500
4,5.000,3.600,1.400


,sepal_length,sepal_width,petal_length,petal_width,species
83,6.000,2.700,5.100,1.600,Iris-versicolor
100,6.300,3.300,6.000,2.500,Iris-virginica
101,5.800,2.700,5.100,1.900,Iris-virginica
102,7.100,3.000,5.900,2.100,Iris-virginica
103,6.300,2.900,5.600,1.800,Iris-virginica


## 5. Aggregate by species

`groupby` helps compare each class in the dataset.

In [14]:
species_summary = iris.groupby('species').agg(
    sepal_length_mean=('sepal_length', 'mean'),
    sepal_length_min=('sepal_length', 'min'),
    sepal_length_max=('sepal_length', 'max'),
    sepal_width_mean=('sepal_width', 'mean'),
    petal_length_mean=('petal_length', 'mean'),
    petal_width_mean=('petal_width', 'mean'),
)
display(species_summary)

,sepal_length_mean,sepal_length_min,sepal_length_max,sepal_width_mean,petal_length_mean,petal_width_mean
species,,,,,,
Iris-setosa,5.006,4.300,5.800,3.418,1.464,0.244
Iris-versicolor,5.936,4.900,7.000,2.770,4.260,1.326
Iris-virginica,6.588,4.900,7.900,2.974,5.552,2.026


In [15]:
display(iris.groupby('species')[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].median())

,sepal_length,sepal_width,petal_length,petal_width
species,,,,
Iris-setosa,5.000,3.400,1.500,0.200
Iris-versicolor,5.900,2.800,4.350,1.300
Iris-virginica,6.500,3.000,5.550,2.000


## 6. Sort and rank

Sorting and ranking are useful when you want to spot extremes quickly.

In [16]:
display(iris.sort_values('petal_length', ascending=False).head(10))
display(iris.sort_values(['species', 'sepal_length'], ascending=[True, False]).head(10))

,sepal_length,sepal_width,petal_length,petal_width,species
118,7.700,2.600,6.900,2.300,Iris-virginica
122,7.700,2.800,6.700,2.000,Iris-virginica
117,7.700,3.800,6.700,2.200,Iris-virginica
105,7.600,3.000,6.600,2.100,Iris-virginica
131,7.900,3.800,6.400,2.000,Iris-virginica
107,7.300,2.900,6.300,1.800,Iris-virginica
130,7.400,2.800,6.100,1.900,Iris-virginica
109,7.200,3.600,6.100,2.500,Iris-virginica
135,7.700,3.000,6.100,2.300,Iris-virginica
100,6.300,3.300,6.000,2.500,Iris-virginica


,sepal_length,sepal_width,petal_length,petal_width,species
14,5.800,4.000,1.200,0.200,Iris-setosa
15,5.700,4.400,1.500,0.400,Iris-setosa
18,5.700,3.800,1.700,0.300,Iris-setosa
33,5.500,4.200,1.400,0.200,Iris-setosa
36,5.500,3.500,1.300,0.200,Iris-setosa
5,5.400,3.900,1.700,0.400,Iris-setosa
10,5.400,3.700,1.500,0.200,Iris-setosa
16,5.400,3.900,1.300,0.400,Iris-setosa
20,5.400,3.400,1.700,0.200,Iris-setosa
31,5.400,3.400,1.500,0.400,Iris-setosa


## 7. Create derived columns

Feature engineering with pandas is usually just a few vectorized operations.

In [17]:
iris = iris.copy()
iris['sepal_area'] = iris['sepal_length'] * iris['sepal_width']
iris['petal_area'] = iris['petal_length'] * iris['petal_width']
iris['length_width_ratio'] = iris['petal_length'] / iris['petal_width']
display(iris.head())

,sepal_length,sepal_width,petal_length,petal_width,species,sepal_area,petal_area,length_width_ratio
0,5.100,3.500,1.400,0.200,Iris-setosa,17.850,0.280,7.000
1,4.900,3.000,1.400,0.200,Iris-setosa,14.700,0.280,7.000
2,4.700,3.200,1.300,0.200,Iris-setosa,15.040,0.260,6.500
3,4.600,3.100,1.500,0.200,Iris-setosa,14.260,0.300,7.500
4,5.000,3.600,1.400,0.200,Iris-setosa,18.000,0.280,7.000


In [18]:
display(iris[['sepal_area', 'petal_area', 'length_width_ratio']].describe())

,sepal_area,petal_area,length_width_ratio
count,150.000,150.000,150.000
mean,17.807,5.793,4.367
std,3.369,4.713,2.652
min,10.000,0.110,2.125
25%,15.645,0.420,2.802
50%,17.660,5.615,3.300
75%,20.325,9.690,4.667
max,30.020,15.870,15.000


## 8. Correlations and relationships

Correlation is a simple way to inspect how numeric fields move together.

In [19]:
numeric_columns = iris.select_dtypes(include='number')
display(numeric_columns.corr())

,sepal_length,sepal_width,petal_length,petal_width,sepal_area,petal_area,length_width_ratio
sepal_length,1.000,-0.109,0.872,0.818,0.683,0.857,-0.563
sepal_width,-0.109,1.000,-0.421,-0.357,0.645,-0.281,0.321
petal_length,0.872,-0.421,1.000,0.963,0.367,0.958,-0.684
petal_width,0.818,-0.357,0.963,1.000,0.375,0.980,-0.734
sepal_area,0.683,0.645,0.367,0.375,1.000,0.459,-0.204
petal_area,0.857,-0.281,0.958,0.980,0.459,1.000,-0.651
length_width_ratio,-0.563,0.321,-0.684,-0.734,-0.204,-0.651,1.000


In [20]:
display(iris.groupby('species')[['sepal_area', 'petal_area', 'length_width_ratio']].mean())

,sepal_area,petal_area,length_width_ratio
species,,,
Iris-setosa,17.209,0.363,7.078
Iris-versicolor,16.526,5.720,3.243
Iris-virginica,19.685,11.296,2.781


## 9. Crosstab and pivot table

These are useful when you want a compact summary table.

In [21]:
iris['petal_length_band'] = pd.cut(iris['petal_length'], bins=[0, 2, 4, 7], labels=['short', 'medium', 'long'])
display(pd.crosstab(iris['species'], iris['petal_length_band']))
display(pd.pivot_table(iris, index='species', values=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'], aggfunc='mean'))

petal_length_band,short,medium,long
species,,,
Iris-setosa,50,0,0
Iris-versicolor,0,16,34
Iris-virginica,0,0,50


,petal_length,petal_width,sepal_length,sepal_width
species,,,,
Iris-setosa,1.464,0.244,5.006,3.418
Iris-versicolor,4.260,1.326,5.936,2.770
Iris-virginica,5.552,2.026,6.588,2.974


## 10. Sampling and export

`sample()` is helpful for quick spot checks, and `to_csv()` makes it easy to export results.

In [22]:
display(iris.sample(8, random_state=7))
csv_preview = iris.head(5).to_csv(index=False)
print(csv_preview)

,sepal_length,sepal_width,petal_length,petal_width,species,sepal_area,petal_area,length_width_ratio,petal_length_band
149,5.900,3.000,5.100,1.800,Iris-virginica,17.700,9.180,2.833,long
84,5.400,3.000,4.500,1.500,Iris-versicolor,16.200,6.750,3.000,long
40,5.000,3.500,1.300,0.300,Iris-setosa,17.500,0.390,4.333,short
66,5.600,3.000,4.500,1.500,Iris-versicolor,16.800,6.750,3.000,long
106,4.900,2.500,4.500,1.700,Iris-virginica,12.250,7.650,2.647,long
41,4.500,2.300,1.300,0.300,Iris-setosa,10.350,0.390,4.333,short
52,6.900,3.100,4.900,1.500,Iris-versicolor,21.390,7.350,3.267,long
94,5.600,2.700,4.200,1.300,Iris-versicolor,15.120,5.460,3.231,long


sepal_length,sepal_width,petal_length,petal_width,species,sepal_area,petal_area,length_width_ratio,petal_length_band
5.1,3.5,1.4,0.2,Iris-setosa,17.849999999999998,0.27999999999999997,6.999999999999999,short
4.9,3.0,1.4,0.2,Iris-setosa,14.700000000000001,0.27999999999999997,6.999999999999999,short
4.7,3.2,1.3,0.2,Iris-setosa,15.040000000000001,0.26,6.5,short
4.6,3.1,1.5,0.2,Iris-setosa,14.26,0.30000000000000004,7.5,short
5.0,3.6,1.4,0.2,Iris-setosa,18.0,0.27999999999999997,6.999999999999999,short



## Summary

This notebook used a real dataset to explore the most common pandas workflows: loading data, inspecting shape and types, handling missing values, selecting rows, filtering, grouping, sorting, creating new columns, using crosstab and pivot tables, and sampling/exporting data.